[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/junlinghu/book-agents/blob/main/tutorial/9-autonomy-policy.ipynb)

## Colab setup

On Google Colab, use the setup in `tutorial/common/colab.py`. A fresh Colab runtime does not have this repository, so clone the repo into the session first. Then, before the lesson cells, run `setup_colab` from that module (import it from `tutorial.common.colab` and call it, or open the file and run `setup_colab` from there). The helper installs the tutorial packages, checks the repo out under `/content/book-agents` when it is still missing, and copies a Colab secret named `OPENAI_API_KEY`. Local Jupyter and VS Code can skip it when you already have the repo.

An API key is required. Set `OPENAI_API_KEY` in Colab secrets (the key icon, secret name `OPENAI_API_KEY`) before you run `setup_colab`. Locally, put the same name in the repository-root `.env`. A missing key stops the notebook. The key is not printed.

Edits you make in Colab stay in that session. They do not push to GitHub.

# 9. Autonomy policy

A tool call is a proposal. The gate decides.

An FAQ lookup is auto. A discount code waits for a person. Charging a card, canceling an order, and emailing a customer never run, even when a token matches that exact call. The token is a hash of the tool name and the arguments. A different code is a different token.

The customer asks to apply `JAR10` and not to charge a card. The first run holds the discount. A second run passes the token from the first.

## Setup

Find the repo so `import tutorial` works, then load the key. `chat` calls the OpenAI Chat Completions API. The key is not printed.

In [ ]:
import sys
from pathlib import Path


def find_root():
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "tutorial" / "common" / "client.py").is_file():
            return folder
    raise RuntimeError("Open this notebook inside the book-agents repository.")


ROOT = find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from tutorial.runtime import require_key

_, model = require_key()
print("model:", model)

## Three tiers, no model

These decisions are ordinary code. The wording of a later chat does not change them.

In [ ]:
import re

import tutorial.common.facts
import tutorial.common.catalog
import tutorial.common.gate  # registers the gate and the side-effect tools

from tutorial.common.autonomy import approval_token
from tutorial.common.gate import format_decision, gate_call, reset_orders
from tutorial.common.get_db import reset_db
from tutorial.common.harness import check, system_text
from tutorial.common.loop import run_agent

reset_db()
reset_orders()
faq = gate_call("get_store_fact", {"topic": "hours"}, set())
print(format_decision(faq))
discount_args = {"code": "JAR10"}
held = gate_call("apply_discount", discount_args, set())
print(format_decision(held))
charge_args = {"amount_cents": 1800}
charge = gate_call("charge_card", charge_args, {approval_token("charge_card", charge_args)})
print(format_decision(charge))
cancel_args = {"order_id": "HJ-1001"}
cancel = gate_call("cancel_order", cancel_args, {approval_token("cancel_order", cancel_args)})
print(format_decision(cancel))
mail = gate_call(
    "send_customer_email",
    {"to": "maya@example.com", "body": "order status"},
    set(),
)
print(format_decision(mail))
check(faq["decision"] == "allowed" and faq["tier"] == "auto", "an FAQ lookup is auto")
check(held["decision"] == "confirm_required", "a discount waits for a person")
check(charge["decision"] == "denied", "a matching token still does not charge a card")
check(cancel["decision"] == "denied", "a matching token still does not cancel an order")
check(mail["decision"] == "denied", "emailing a customer is never sent")

## The customer asks

The first loop should hold `JAR10`. The second loop passes that token and still must not charge a card.

In [ ]:
question = "Please apply discount code JAR10 to my order. Do not charge my card."
first = run_agent(question, system_text())
print("ANSWER:", first["text"])
log = "\n".join(first["tool_log"])
check("apply_discount" in log, "the discount tool was requested")
check("confirm_required" in log.lower(), "the discount was not applied silently")
check("charge_card" not in log or "denied" in log.lower(), "no card was charged")
tokens = re.findall(r"token=([0-9a-f]{8})", log)
print("approval tokens:", tokens)
if tokens:
    second = run_agent(question, system_text(), confirmed_tokens=tokens)
    print("ANSWER:", second["text"])
    check("DISCOUNT HELD" in "\n".join(second["tool_log"]), "the same code runs after the token matches")